# 00 — Exploratory Data Analysis

**Research question:** Does dynamic judge workload at case opening help predict the *procedural complexity* that a case will develop, beyond what basic filing attributes already explain?

This notebook explores the full case-level dataset (`data/case_features.parquet`, 168k cases) to understand:
- The distribution of the **target variable**: `complexity_index`
- The distribution of **filing features** (what we know at case open)
- The role of **judges and their workload**
- Background context on **LOS variance** and why it motivated the research pivot

All visualizations use **Plotly Express** (interactive).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.features import add_derived_columns, FILING_FEATURES_NUMERIC, PARTY_TYPE_FEATURES

df = pd.read_parquet(ROOT / "data" / "case_features.parquet")
df = add_derived_columns(df)
print(f"Dataset: {df.shape[0]:,} cases × {df.shape[1]} columns")
df[["case_type", "los_days", "complexity_index", "n_events", "n_motions"]].describe().round(2)

The dataset contains ~168k closed federal court cases. Each row is one case. The key columns are:

| Column | Role |
|---|---|
| `complexity_index` | **Model target** — z-score composite of 4 complexity metrics |
| `case_type` | `cv` (civil) or `cr` (criminal) — models run separately |
| `los_days` | Case duration in days — background context only |
| `District_Judge` | Anonymised judge ID |
| `case_open_date` | Filing date — defines what's known *at filing* |
| Filing attributes | Party counts, suit type, MDL flag, city — features in Model A |

## 1. Civil vs Criminal split

Civil (`cv`) and criminal (`cr`) cases operate under different procedural rules, have different average durations, and different complexity profiles. We train and evaluate models **separately** for each case type.

In [ ]:
type_counts = df["case_type"].value_counts().rename_axis("case_type").reset_index(name="n_cases")
type_counts["label"] = type_counts["case_type"].map({"cv": "Civil", "cr": "Criminal"})
type_counts["pct"] = (100 * type_counts["n_cases"] / type_counts["n_cases"].sum()).round(1)

fig = px.bar(
    type_counts, x="label", y="n_cases",
    text=type_counts.apply(lambda r: f"{r['n_cases']:,} ({r['pct']}%)", axis=1),
    color="label",
    color_discrete_map={"Civil": "#4C72B0", "Criminal": "#DD8452"},
    title="Case count by type",
    labels={"label": "", "n_cases": "Number of cases"},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_range=[0, type_counts["n_cases"].max() * 1.15])
fig.show()

Civil cases dominate (~90% of the dataset). Criminal cases have structurally higher event counts per case because they include plea hearings, sentencing, and pre-trial motions that don't appear in civil dockets. This structural difference is why the analysis is stratified.

## 2. Target variable: Procedural Complexity Index

`complexity_index` is the **mean of four z-scored metrics** computed after the case closes:

| Metric | Meaning |
|---|---|
| `n_events` | Total docket entries |
| `n_activity_types` | Variety of event types (breadth) |
| `n_motions` | Number of motions filed |
| `activity_entropy` | Shannon entropy of event-type distribution |

By z-scoring each and averaging, the index is zero-centred and unit-comparable across cases. A high value means the case generated unusually many, diverse, and motion-heavy docket events — i.e., it was procedurally complex.

> **Why not predict LOS directly?** The complexity metrics above are *retrospective* — they're measured after the case is done, so predicting LOS from them is circular (endogenous). Instead we predict complexity *from* filing attributes and judge workload — information available *before* any docket activity.

In [ ]:
fig = px.histogram(
    df, x="complexity_index", color="case_type",
    facet_col="case_type", facet_col_spacing=0.08,
    nbins=80, opacity=0.85,
    color_discrete_map={"cv": "#4C72B0", "cr": "#DD8452"},
    title="Distribution of complexity_index by case type",
    labels={"complexity_index": "Complexity index"},
)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("case_type=", "")))
fig.show()

In [ ]:
metrics = ["n_events", "n_activity_types", "n_motions", "activity_entropy"]
titles  = ["Total events", "Unique activity types", "Motions filed", "Activity entropy"]

fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

for i, (m, t) in enumerate(zip(metrics, titles)):
    row, col = divmod(i, 2)
    for ct, color in [("cv", "rgba(76,114,176,0.6)"), ("cr", "rgba(221,132,82,0.6)")]:
        vals = df.loc[df["case_type"] == ct, m].dropna()
        cap  = float(vals.quantile(0.99))
        fig.add_trace(
            go.Histogram(x=vals.clip(upper=cap), name=ct,
                         marker_color=color, showlegend=(i == 0)),
            row=row + 1, col=col + 1,
        )

fig.update_layout(
    title="Raw complexity metrics (clipped at 99th percentile)",
    barmode="overlay", height=520,
    legend=dict(title="case_type", x=0.82, y=0.98),
)
fig.show()

All four metrics are **right-skewed** — the vast majority of cases have modest complexity, while a small tail of cases generates an outsized number of events and motions. Z-scoring within each metric before combining into `complexity_index` prevents the high-variance metrics (especially `n_events`) from dominating the composite.

## 3. LOS — background context

LOS (`los_days`) is not our model target, but understanding its distribution and its variance across judges motivates the research design. The supervisor noted that a raw LOS boxplot by judge does not control for case mix — a judge assigned many MDL or criminal cases will naturally show longer LOS regardless of their behaviour.

In [ ]:
los_clean = df[df["los_days"].between(1, df["los_days"].quantile(0.99))].copy()

fig = px.histogram(
    los_clean, x="los_days", color="case_type",
    facet_col="case_type", facet_col_spacing=0.08,
    nbins=80, opacity=0.85,
    color_discrete_map={"cv": "#4C72B0", "cr": "#DD8452"},
    title="LOS distribution (clipped at 99th percentile)",
    labels={"los_days": "Days"},
)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("case_type=", "")))
fig.show()

In [ ]:
# LOS by judge — raw, uncontrolled (top 25 by case count)
top_judges = df["District_Judge"].value_counts().head(25).index
df_top = df[df["District_Judge"].isin(top_judges) & df["los_days"].between(1, 3000)]

fig = px.box(
    df_top, x="District_Judge", y="los_days",
    color="case_type",
    color_discrete_map={"cv": "#4C72B0", "cr": "#DD8452"},
    title="LOS by judge — raw, uncontrolled (top 25 judges)",
    labels={"los_days": "LOS (days)", "District_Judge": "Judge ID"},
)
fig.update_layout(xaxis_tickangle=45, xaxis_title="")
fig.show()

Visually the LOS distributions appear somewhat similar across judges — but this raw view conflates **case mix** with **judicial behaviour**. A judge with many simple civil cases will look fast; one assigned mostly MDL cases will look slow. The correct test controls for case attributes first, then asks whether residual variation is still systematically linked to the judge (or their workload).

## 4. Filing features (Model A inputs)

These are the features available **at case filing** — before any docket events occur. They form the baseline (Model A). Model B adds `judge_workload_at_open` on top of these.

Key feature groups:
- **Plaintiff / Defendant attributes**: counts, shares (pro-se, pro-hac-vice), counsel counts
- **Party types**: amicus, intervenor, trustee, etc.
- **Case flags**: `is_cv`, `is_mdl`
- **Suit structure**: derived from `nature_suits` (entropy, dominant type, presence flags)
- **City**: district location (one-hot encoded)

In [ ]:
filing_cols = [c for c in FILING_FEATURES_NUMERIC if c in df.columns]
missing_pct = (df[filing_cols].isna().mean() * 100).sort_values(ascending=False)

fig = px.bar(
    x=missing_pct.values, y=missing_pct.index,
    orientation="h",
    title="Missing values (%) in filing features",
    labels={"x": "Missing (%)", "y": ""},
    color=missing_pct.values,
    color_continuous_scale="Reds",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
key_features = [
    "plaintiffs_count", "Defendants_count",
    "related_case_count", "plaintiffs_counsels_count"
]
fig = make_subplots(rows=2, cols=2, subplot_titles=key_features)

for i, feat in enumerate(key_features):
    row, col = divmod(i, 2)
    vals = df[feat].dropna().clip(upper=df[feat].quantile(0.99))
    fig.add_trace(
        go.Histogram(x=vals, marker_color="#4C72B0", showlegend=False),
        row=row + 1, col=col + 1,
    )

fig.update_layout(title="Key filing features — distribution (clipped at 99th pct)", height=500)
fig.show()

Most filing features are highly right-skewed: the typical case has 1–2 plaintiffs, 1 defendant, no related cases. This long tail is handled naturally by tree-based models (Random Forest and XGBoost) without needing log-transforms.

## 5. MDL cases

Multi-District Litigation (`is_mdl = True`) cases are procedurally unique — they consolidate claims from multiple districts and tend to generate far more events. The `is_mdl` flag is expected to be one of the strongest predictors of complexity.

In [ ]:
df["MDL"] = df["is_mdl"].map({True: "MDL", False: "Non-MDL"})

fig = px.box(
    df, x="MDL", y="complexity_index",
    color="case_type", facet_col="case_type",
    color_discrete_map={"cv": "#4C72B0", "cr": "#DD8452"},
    title="Complexity index: MDL vs Non-MDL",
    labels={"complexity_index": "Complexity index", "MDL": ""},
)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("case_type=", "")))
fig.show()

MDL cases are substantially more complex across both civil and criminal types. This confirms `is_mdl` must be included in Model A.

## 6. Complexity by judge — motivation for the workload hypothesis

We check whether `complexity_index` visibly varies across judges, and whether this variation persists even after stratifying by case type. If judges with heavier dockets handle cases differently — e.g., shorter hearings, fewer scheduling opportunities — that would show up as a workload effect on complexity.

In [ ]:
# Filter to judges with >= 30 cases; take top 25 by case count
judge_counts = df["District_Judge"].value_counts()
judges_30    = judge_counts[judge_counts >= 30].head(25).index
df_j = df[df["District_Judge"].isin(judges_30)]

fig = px.box(
    df_j, x="District_Judge", y="complexity_index",
    color="case_type",
    color_discrete_map={"cv": "#4C72B0", "cr": "#DD8452"},
    title="Complexity index by judge (top 25, ≥ 30 cases each)",
    labels={"complexity_index": "Complexity index", "District_Judge": ""},
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

In [ ]:
# Kruskal-Wallis test: do complexity distributions differ across judges?
groups_cv = [
    g["complexity_index"].dropna().values
    for _, g in df_j[df_j["case_type"] == "cv"].groupby("District_Judge")
    if len(g) >= 5
]
groups_cr = [
    g["complexity_index"].dropna().values
    for _, g in df_j[df_j["case_type"] == "cr"].groupby("District_Judge")
    if len(g) >= 5
]

if len(groups_cv) >= 2:
    h_cv, p_cv = kruskal(*groups_cv)
    print(f"Civil   — Kruskal-Wallis H={h_cv:.2f}, p={p_cv:.4f}")
if len(groups_cr) >= 2:
    h_cr, p_cr = kruskal(*groups_cr)
    print(f"Criminal — Kruskal-Wallis H={h_cr:.2f}, p={p_cr:.4f}")
print()
print("Note: with 168k cases, even small differences will be statistically")
print("significant. The key question is whether workload EXPLAINS some of that")
print("variation beyond case-type composition.")

The Kruskal-Wallis test confirms that complexity distributions differ significantly across judges — but as noted, this likely conflates case mix with judge effects. The model comparison (Model A vs Model B) isolates the workload contribution.

## 7. Cases per judge

Many judges appear in only a few cases — their data is too sparse for reliable workload estimation. We apply a minimum threshold of 30 cases per judge throughout the analysis.

In [ ]:
cases_per_judge = df["District_Judge"].value_counts()

fig = px.histogram(
    x=cases_per_judge.values, nbins=40,
    title="Cases per judge — distribution",
    labels={"x": "Number of cases", "y": "Number of judges"},
    color_discrete_sequence=["#4C72B0"],
)
fig.add_vline(x=30, line_dash="dash", line_color="crimson",
              annotation_text="min threshold (30)",
              annotation_position="top right")
fig.show()

n_above = (cases_per_judge >= 30).sum()
print(f"Judges with >= 30 cases: {n_above} / {len(cases_per_judge)}")

## 8. Correlation of filing features with complexity

Linear Pearson correlations give a first-pass view of which filing-time attributes are associated with eventual complexity. Tree models will capture non-linear interactions too, but the correlation chart identifies the most promising individual predictors.

In [ ]:
corr_cols = [c for c in FILING_FEATURES_NUMERIC + PARTY_TYPE_FEATURES if c in df.columns]
corr = (
    df[corr_cols + ["complexity_index"]]
    .corr()[["complexity_index"]]
    .drop("complexity_index")
    .sort_values("complexity_index")
)

fig = px.bar(
    x=corr["complexity_index"].values,
    y=corr.index,
    orientation="h",
    title="Pearson correlation with complexity_index (filing features)",
    labels={"x": "Pearson r", "y": ""},
    color=corr["complexity_index"].values,
    color_continuous_scale="RdBu",
    range_color=[-0.4, 0.4],
)
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(coloraxis_showscale=False)
fig.show()

Individual linear correlations are modest (typically |r| < 0.2), which is expected for procedural complexity — no single filing attribute captures it well. This motivates using tree-based models that combine many weak signals through non-linear interactions.

## 9. Temporal trends

Cases span multiple years. Temporal drift in complexity or judge assignment is a potential confounder — if judge workload grew over time (e.g., increasing docket sizes), naive comparisons may attribute year effects to workload. The temporal train/test split in the models (80th percentile of open date as cutoff) accounts for this.

In [ ]:
df["year"] = pd.to_datetime(df["case_open_date"]).dt.year
yearly = (
    df.groupby(["year", "case_type"])["complexity_index"]
    .agg(["median", "count"])
    .reset_index()
    .rename(columns={"median": "median_complexity", "count": "n_cases"})
)

fig = make_subplots(specs=[[{"secondary_y": True}]])

for ct, color in [("cv", "#4C72B0"), ("cr", "#DD8452")]:
    sub = yearly[yearly["case_type"] == ct]
    fig.add_trace(go.Scatter(x=sub["year"], y=sub["median_complexity"],
                             mode="lines+markers", name=f"{ct} complexity",
                             line_color=color), secondary_y=False)
    fig.add_trace(go.Bar(x=sub["year"], y=sub["n_cases"],
                         name=f"{ct} cases", marker_color=color,
                         opacity=0.2, showlegend=True), secondary_y=True)

fig.update_layout(title="Median complexity index over time (with case volume)")
fig.update_yaxes(title_text="Median complexity index", secondary_y=False)
fig.update_yaxes(title_text="Number of cases", secondary_y=True)
fig.show()

## Summary

| Finding | Implication for modelling |
|---|---|
| Civil and criminal cases differ structurally | Train models separately for `cv` and `cr` |
| `complexity_index` is right-skewed with heavy tails | Z-score composite is a robust target; tree models handle skew well |
| MDL cases are dramatically more complex | `is_mdl` is an essential Model A feature |
| Raw LOS boxplot by judge does not control for case mix | The correct test is model-based: control for filing attributes first |
| Kruskal-Wallis confirms inter-judge complexity variation | Motivates adding workload as Model B feature |
| Linear correlations of filing features with complexity are modest | Non-linear tree models are the right choice |
| Complexity shows temporal drift | Temporal train/test split (80/20 by open date) prevents leakage |

**Next steps:** Engineer `judge_workload_at_open` (number of concurrently open cases per judge at filing date), then compare Model A vs Model B.